# Preprocessing v3 — Données de matchs League of Legends

## Objectif
Ce notebook prépare les données brutes Riot API pour la modélisation ML.

## Étapes du pipeline
1. Chargement des données recap/timeline + profils joueurs
2. Filtrage des matchs invalides
3. Flattening des informations minute/joueur/match
4. Feature engineering (rang, région, champion/rôle)
5. Nettoyage final (corrélations, colonnes constantes, NaN)
6. Export des datasets prétraités

## Convention de cible
- `win = 1` : victoire équipe bleue
- `win = 0` : victoire équipe rouge

## 1. Librairies et prérequis

Exécuter le notebook dans l'ordre (Run All) pour garantir la cohérence des variables en mémoire.

In [1]:
import os 
import json
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Récupération des données 

In [2]:
region = ["americas","asia","europe"]
puuids = []

dossier = "data"

for nom_fichier in sorted(os.listdir(dossier)):
    if nom_fichier.startswith("puuid_summoners_") and nom_fichier.endswith(".json"):
        chemin = os.path.join(dossier, nom_fichier)
        with open(chemin, "r", encoding="utf-8") as f:
            contenu = json.load(f)
            for r in region :
                puuids.extend(contenu[r])

print(f"{len(puuids)} puuids chargés")

9000 puuids chargés


In [3]:
matchs_recap = []
matchs_timeline = []

for region in region:
    region_path = os.path.join("data", region)
    if not os.path.exists(region_path):
        continue
    for folder in os.listdir(region_path):
        folder_path = os.path.join(region_path, folder)
        recap_path = os.path.join(folder_path,"recap.json")
        timeline_path = os.path.join(folder_path,"timeline.json")

        with open(recap_path, "r", encoding="utf-8") as f:
            match_recap = json.load(f)
        with open(timeline_path, "r", encoding="utf-8") as g:
            match_timeline = json.load(g)
        
        matchs_recap.append(match_recap)
        matchs_timeline.append(match_timeline)

print(f"matchs recap chargés : {len(matchs_recap)}")
print(f"matchs timeline chargés : {len(matchs_timeline)}")

matchs recap chargés : 6111
matchs timeline chargés : 6111


In [4]:
for i in range(len(matchs_recap)):
    if matchs_recap[i]["metadata"]["matchId"] != matchs_timeline[i]["metadata"]["matchId"] : 
        print(f"error match n°{i} differents dans les 2 listes")

## 2. Filtrage des matchs à exclure

On supprime les matchs non exploitables pour l'entraînement :
- fin de partie non standard (`endOfGameResult != GameComplete` , les matchs ayant eu une interruption inattendu et donc qui n'ont pas pu finir)
- durée insuffisante par rapport à la fenêtre temporelle choisie

In [5]:
def matchs_interrompu(matchs_recap):
    """
    args:
        matchs_recap (list): liste des recaps de tous les matchs chargés
    return:
        set: ensemble des matchs à supprimer (matchs ayant eu une interruption inattendu et donc qui n'ont pas pu finir)
    goal:
        identifier les matchs à supprimer (matchs ayant eu une interruption inattendu et donc qui n'ont pas pu finir)
    """
    matchs_2_supp = set()
    i = 0
    error = set()
    match_error = set()
    for m in range(len(matchs_recap)) :
        match_recap = matchs_recap[m]
        if match_recap["info"]["endOfGameResult"] != "GameComplete":
            print(f"Ce match a eu une interruption inattendu : {match_recap["metadata"]["matchId"]} ")
            print(f"endOfGameResult = {match_recap["info"]["endOfGameResult"]}")
            error.add(match_recap["info"]["endOfGameResult"])
            #match_error.add(match_recap["metadata"]["matchId"])
            matchs_2_supp.add(m)
            i+=1
    print(f"Nombre match interrompu : {i}")
    print(f"type d'erreur rencontrée : {error}")
    return matchs_2_supp 


In [6]:
def match_less_than_time_needed(matchs_timeline,time):
    """
    args:
        matchs_timeline (list): liste des timelines de tous les matchs chargés
        time (int): durée minimale en minutes pour qu'un match soit considéré comme exploitable
    return:
        set: ensemble des matchs à supprimer (matchs ayant durés moins de time minutes)
    goal:
        identifier les matchs à supprimer (matchs ayant durés moins de time minutes)
    """
    i = 0 
    matchs_2_supp = set()
    #match_less_15 = set()
    for m in range(len(matchs_timeline)) : 
        match_timeline = matchs_timeline[m]# ou bien gameDuration dans matchs_recap . 1 frame correspond a 1min de jeu dans timeline
        if len(match_timeline['info']['frames'])<time : 
            #print(f"Ce match est terminé avant 15min : {match["metadata"]["matchId"]} ")
            #match_less_15.add(match_timeline["metadata"]["matchId"])
            matchs_2_supp.add(m)
            i+=1

    print(f"Nombre match de moins de {time}min : {i}")
    return matchs_2_supp

In [7]:

def recup_matchs_without_pb(matchs_timeline,matchs_recap,matchs_2_supp):  
    """
    args:
        matchs_timeline (list): liste des timelines de tous les matchs chargés
        matchs_recap (list): liste des recaps de tous les matchs chargés
        matchs_2_supp (set): ensemble des index des matchs à supprimer (matchs ayant eu une interruption inattendu et donc qui n'ont pas pu finir, ainsi que les matchs ayant durés moins de 15min)
    return:
        list: liste des recaps de tous les matchs exploitables
        list: liste des timelines de tous les matchs exploitables
    goal:
        identifier les matchs à supprimer (matchs ayant eu une interruption inattendu et donc qui n'ont pas pu finir, ainsi que les matchs ayant durés moins de 15min) et renvoyer les listes de matchs sans ces matchs à problèmes
    """
    matchs_recap = [m for i, m in enumerate(matchs_recap) if i not in matchs_2_supp]
    matchs_timeline = [m for i, m in enumerate(matchs_timeline) if i not in matchs_2_supp]

    print(f"matchs recap chargés : {len(matchs_recap)}")
    print(f"matchs timeline chargés : {len(matchs_timeline)}")
    return matchs_recap,matchs_timeline

In [8]:
def match_to_df(match):
    """
    args:
        match (dict): dictionnaire contenant les données d'un match (timeline)
    return:
        DataFrame: dataframe contenant les données du match minute par minute pour chaque participant (10 lignes par minute)
    goal:
        transformer les données d'un match (timeline) en un dataframe exploitable pour l'entraînement
    """
    rows_frames = []
    rows_events = []
    frames = match['info']['frames']
    
    joueursIDs = ['1','2','3','4','5','6','7','8','9','10']
    col_keep_participantsFrames = {
    'damageStats' : ["totalDamageDoneToChampions","totalDamageTaken"],
    "totalGold" : [],
    "xp": [],
    "level" : [],
    "minionsKilled" : [],
    "jungleMinionsKilled" : [],
    }

    events_to_keep = ['BUILDING_KILL','CHAMPION_KILL','ELITE_MONSTER_KILL','TURRET_PLATE_DESTROYED']

    for frame in frames:
        minute = frame['timestamp'] // 60000 #60000ms = 1min 
        participantsFrames = frame['participantFrames']
        events = frame['events']

        for pid in joueursIDs:
            pf = participantsFrames.get(pid, {})
            row = {'minute': minute, 'participantId': int(pid)}

            for col, subkeys in col_keep_participantsFrames.items():
                if subkeys:
                    nested = pf.get(col, {})
                    for subkey in subkeys:
                        row[subkey] = nested.get(subkey, 0)
                else:
                    row[col] = pf.get(col, 0)

            rows_frames.append(row)

        event_cols = ['kills', 'deaths', 'assists', 'bounty_collected',
                      'shutdown_bounty_collected', 'buildings_killed',
                      'elite_monsters_killed', 'turret_plates_destroyed']

        def make_event_row(pid):
            return {'minute': minute, 'participantId': pid,
                    **{col: 0 for col in event_cols}}

        for event in events:
            event_type = event.get('type')
            if event_type not in events_to_keep:
                continue

            if event_type == 'CHAMPION_KILL':
                killer_id = event.get('killerId', 0)
                victim_id = event.get('victimId', 0)
                assistants = event.get('assistingParticipantIds', [])
                if killer_id > 0:
                    r = make_event_row(killer_id)
                    r['kills'] = 1
                    r['bounty_collected'] = event.get('bounty', 0)
                    r['shutdown_bounty_collected'] = event.get('shutdownBounty', 0)
                    rows_events.append(r)
                if victim_id > 0:
                    r = make_event_row(victim_id)
                    r['deaths'] = 1
                    rows_events.append(r)
                for assist_id in assistants:
                    r = make_event_row(assist_id)
                    r['assists'] = 1
                    rows_events.append(r)

            elif event_type == 'BUILDING_KILL':
                killer_id = event.get('killerId', 0)
                if killer_id > 0:
                    r = make_event_row(killer_id)
                    r['buildings_killed'] = 1
                    rows_events.append(r)

            elif event_type == 'ELITE_MONSTER_KILL':
                killer_id = event.get('killerId', 0)
                if killer_id > 0:
                    r = make_event_row(killer_id)
                    r['elite_monsters_killed'] = 1
                    rows_events.append(r)

            elif event_type == 'TURRET_PLATE_DESTROYED':
                killer_id = event.get('killerId', 0)
                if killer_id > 0:
                    r = make_event_row(killer_id)
                    r['turret_plates_destroyed'] = 1
                    rows_events.append(r)

    # Dédupliquer les frames en gardant la dernière par (minute, participantId)
    df_frames = (
        pd.DataFrame(rows_frames)
        .sort_values('minute')
        .drop_duplicates(subset=['minute', 'participantId'], keep='last')
    )

    event_cols = ['kills', 'deaths', 'assists', 'bounty_collected',
                  'shutdown_bounty_collected', 'buildings_killed',
                  'elite_monsters_killed', 'turret_plates_destroyed']

    df_events = (
        pd.DataFrame(rows_events)
        .groupby(['minute', 'participantId'])
        .sum()
        .reset_index()
    )

    all_minutes = df_frames[['minute', 'participantId']]
    df_events = all_minutes.merge(df_events, on=['minute', 'participantId'], how='left').fillna(0)

    df_events[event_cols] = (
        df_events
        .sort_values(['participantId', 'minute'])
        .groupby('participantId')[event_cols]
        .cumsum()
    )

    df = df_frames.merge(df_events, on=['minute', 'participantId'], how='left')

    return df.sort_values(['minute', 'participantId']).reset_index(drop=True)


In [9]:
# Ordre canonique des roles : p0=TOP, p1=JGL, p2=MID, p3=BOT, p4=SUP (equipe bleue)
#                              p5=TOP, p6=JGL, p7=MID, p8=BOT, p9=SUP (equipe rouge)
ROLE_ORDER = {"TOP": 0, "JUNGLE": 1, "MIDDLE": 2, "BOTTOM": 3, "UTILITY": 4}

# renvoie un df avec les minutes qui nous interessent : chaque ligne contient un match entier
def applatir_matchs(matchs, matchs_recap, puuids, minutes_select=[5, 10, 15]):
    """
    args:
        matchs (list): liste des timelines de tous les matchs chargés
        matchs_recap (list): liste des recaps de tous les matchs chargés
        puuids (list): liste des puuids de tous les joueurs chargés
        minutes_select (list or int): liste des minutes à sélectionner pour l'aplatissement (ex: [5,10,15]) ou int si on veut une seule minute (ex: 10)
    return:
        DataFrame: dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
    goal:
        transformer les données de tous les matchs sélectionnés en un dataframe exploitable pour l'entraînement
    """
    all_rows = []

    for i, match in enumerate(matchs):
        df_match = match_to_df(match)
        match_id = match['metadata']['matchId']

        # Récupérer le winningTeam depuis l'event GAME_END
        winning_team = None
        for frame in match['info']['frames']:
            for event in frame['events']:
                if event.get('type') == 'GAME_END':
                    winning_team = event.get('winningTeam')
                    break
            if winning_team is not None:
                break

        # Garder uniquement les minutes voulues
        minutes_list = [minutes_select] if isinstance(minutes_select, int) else minutes_select
        df_filtered = df_match[df_match['minute'].isin(minutes_list)]
        feature_cols = df_filtered.columns.difference(['minute', 'participantId'])

        row = {'matchId': match_id, 'winningTeam': winning_team}

        # Garder le rank et le tier d'un joueur
        for puuid in matchs_recap[i]["metadata"]["participants"]:
            for j in range(len(puuids)):
                if puuid == puuids[j]["puuid"]:
                    row["rank"] = puuids[j]["rank"]
                    row["tier"] = puuids[j]["tier"]

        # Construire le mapping participantId -> position (0-9) trie par role
        # Blue team (teamId=100) : participantId 1-5  -> positions 0-4
        # Red team  (teamId=200) : participantId 6-10 -> positions 5-9
        participants = matchs_recap[i]["info"]["participants"]
        pid_to_pos = {}

        for team_offset, team_parts in [(0, participants[:5]), (5, participants[5:])]:
            sorted_parts = sorted(team_parts, key=lambda p: ROLE_ORDER.get(p.get('teamPosition', ''), 99))
            for pos_in_team, p in enumerate(sorted_parts):
                pid_to_pos[p['participantId']] = team_offset + pos_in_team

        # Champion & role avec les positions triees par role
        for pid, pos in pid_to_pos.items():
            p = next(pt for pt in participants if pt['participantId'] == pid)
            row[f"p{pos}_champion"] = p["championName"]
            row[f"p{pos}_role"]     = p["teamPosition"]

        # Stats : on utilise pid_to_pos pour aligner avec les champion/role
        for _, r in df_filtered.iterrows():
            pid  = int(r['participantId'])
            pos  = pid_to_pos.get(pid, pid - 1)  # fallback : pid-1 si pid absent
            min_ = int(r['minute'])
            for col in feature_cols:
                row[f"p{pos}_{col}_min{min_}"] = r[col]

        all_rows.append(row)

    df_final = pd.DataFrame(all_rows)
    return df_final


## 3. Encodage et Feature Engineering

Les fonctions d'encodage transforment les variables catégoriques en représentations numériques adaptées au machine learning :

In [10]:
def encode_rank(row):
    """
    args:
        row (Series): une ligne du dataframe matchs_recap contenant les colonnes 'tier' et 'rank'
    return:
        int: un entier représentant le rang de la partie, calculé à partir du tier et du rank
    goal:
        encoder le rang de chaque partie du df matchs_recap en un entier pour pouvoir l'utiliser
    """
    tier_order = {"BRONZE": 0, "SILVER": 4, "GOLD": 8, "PLATINUM": 12, "DIAMOND": 16}
    rank_order = {"I": 3, "II": 2, "III": 1, "IV": 0}
    return tier_order[row['tier']] + rank_order[row['rank']]


In [11]:
def get_region(match_id):
    """
    args:
        match_id (str): identifiant d'un match, dont les 2 premiers caractères correspondent à la région du match (ex: "NA1_1234567890" pour un match de la région NA)
    return:
        str: le préfixe de la région du match (ex: "NA" pour un match de la région NA)
    goal:
        extraire le préfixe de la région du match à partir de son identifiant pour
    """
    prefix = match_id[:2]
    return prefix #NA, EU, KR


In [12]:
def encode_champ_role(df_matchs):
    """
    args:
        df_matchs (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
    return:
        DataFrame: dataframe contenant les mêmes données que df_matchs, mais avec des colonnes supplémentaires pour les winrates de chaque champion par rôle, calculés à partir des données d'entraînement
    goal:
        encoder les colonnes champion et role de df_matchs en fonction du winrate de chaque champion par rapport à son rôle dans notre jeu de données, en calculant ces winrates uniquement àpartir des données d'entraînement pour éviter les fuites de données (data leakage)
    """
    target_col = "win" # On cherche à prédire si l'équipe Bleue gagne
    df_matchs[target_col] = df_matchs[target_col].astype(int)

    # 1. Séparation stricte pour ne pas calculer le winrate sur nos données de test
    train_idx, test_idx = train_test_split(df_matchs.index, test_size=0.2, random_state=42, stratify=df_matchs[target_col])
    global_mean = df_matchs.loc[train_idx, target_col].mean()
    smoothing = 20 # Facteur de régularisation pour les champions peu joués

    roles_order = ["TOP", "JUNGLE", "MIDDLE", "BOTTOM", "UTILITY"]
    player_columns = [(f"p{i}", "blue") for i in range(5)] + [(f"p{i+5}", "red") for i in range(5)]

    def normalize_role(s):
        """
        args:
            s (str): une chaîne de caractères représentant un rôle de joueur dans League of Legends,
                        qui peut être dans différents formats (ex: "MID", "MIDDLE", "ADC", "DUO_CARRY", etc.)
        return:
            str: une chaîne de caractères représentant le rôle normalisé du joueur, selon une convention commune (ex: "MIDDLE" pour les rôles de midlane, "BOTTOM" pour les rôles de botlane, "UTILITY" pour les rôles de support, etc.)
        goal:
            normaliser les différentes façons de représenter les rôles des joueurs dans League of Legends en utilisant une convention commune, pour pouvoir calculer les winrates par champion et par rôle de manière cohérente et éviter les problèmes de dispersion des données (data fragmentation) liés à la multiplicité des formats de rôles possibles
        """
        mapping = {
            "MID": "MIDDLE", "BOT": "BOTTOM", "ADC": "BOTTOM",
            "DUO_CARRY": "BOTTOM", "SUPPORT": "UTILITY", "DUO_SUPPORT": "UTILITY"
        }
        s = str(s).upper()
        if s in ["NAN", "NONE"]:
            return None
        return mapping.get(s, s)

    # Extraction et construction de la base de Winrates
    data_intermediaire = []
    for prefix, team in player_columns:
        champ_col = f"{prefix}_champion"
        role_candidates = [f"{prefix}_teamPosition", f"{prefix}_role", f"{prefix}_lane"]
        role_col = next((c for c in role_candidates if c in df_matchs.columns), None)

        if champ_col in df_matchs.columns and role_col is not None:
            tmp = pd.DataFrame(index=df_matchs.index)
            tmp["champion"] = df_matchs[champ_col].astype(str)
            tmp["role"] = df_matchs[role_col].apply(normalize_role)
            tmp["blue_win"] = df_matchs[target_col]

            tmp = tmp[tmp["role"].isin(roles_order)].copy()
            if tmp.empty:
                continue

            tmp["equipe"] = team
            tmp["match_idx"] = tmp.index
            # Si c'est l'équipe rouge, sa victoire est l'inverse de la victoire bleue
            tmp["won"] = tmp["blue_win"] if team == "blue" else 1 - tmp["blue_win"]
            tmp["champ_role"] = tmp["champion"] + "__" + tmp["role"]

            data_intermediaire.append(tmp[["match_idx", "equipe", "role", "champ_role", "won"]])

    wr_features_computed = False 

    if len(data_intermediaire) == 0:
        print("Aucune colonne champion/rôle exploitable trouvée : features de winrate non ajoutées.")
    else:
        long_df = pd.concat(data_intermediaire, axis=0)

        # Calcul STATISTIQUE EXCLUSIVEMENT sur le train_idx
        long_train = long_df[long_df["match_idx"].isin(train_idx)]
        stats = long_train.groupby("champ_role")["won"].agg(["mean", "count"]).rename(columns={"mean": "wr_mean", "count": "wr_count"})

        # Lissage du winrate (Évite de donner 100% de WR à un champion joué 1 seule fois)
        stats["wr_smooth"] = ((stats["wr_mean"] * stats["wr_count"] + global_mean * smoothing) / (stats["wr_count"] + smoothing))
        wr_map = stats["wr_smooth"].to_dict()

        # Application du dictionnaire
        long_df["champ_role_wr"] = long_df["champ_role"].map(wr_map).fillna(global_mean)
        long_df["feature_name"] = long_df["equipe"] + "_" + long_df["role"].str.lower() + "_champ_wr"

        # Pivot pour remettre en format "1 ligne = 1 match"
        wr_features = long_df.pivot_table(index="match_idx", columns="feature_name", values="champ_role_wr", aggfunc="first")
        wr_features_computed = True   # ← on passe à True seulement si le else s'exécute


        # On intègre ces nouvelles variables puissantes au DataFrame principal
    # 1) Ajout wr_features
    if wr_features_computed: 
        add_cols = [c for c in wr_features.columns if c not in df_matchs.columns]
        if add_cols:
            df_matchs = df_matchs.join(wr_features[add_cols], how="left")
            print(f"wr_features ajoutées: {len(add_cols)}")
        else:
            print("wr_features déjà présentes dans df_matchs")
    else:
        print("wr_features non trouvée: exécute d'abord la cellule de construction wr_features")

    # 1b) Warn si wr_features constantes (souvent signe de fallback global_mean)
    wr_cols = [c for c in df_matchs.columns if c.endswith("_champ_wr")]
    wr_constant_cols = [c for c in wr_cols if df_matchs[c].nunique(dropna=False) <= 1]
    if wr_constant_cols:
        print(f"Avertissement: wr_features constantes détectées ({len(wr_constant_cols)}).")
        print("=> Vérifie que la cellule de création wr_features a bien trouvé les colonnes *_championName / *_teamPosition.")

    #df_matchs = df_matchs.join(wr_features, how="left").fillna(global_mean)
    print(f"Features de Winrate ajoutées. Nouvelle shape : {df_matchs.shape}")
    df_matchs = df_matchs.drop(columns=['p0_champion', 'p0_role', 'p1_champion', 'p1_role', 'p2_champion', 'p2_role', 'p3_champion', 'p3_role', 'p4_champion', 'p4_role', 'p5_champion', 'p5_role', 'p6_champion', 'p6_role', 'p7_champion', 'p7_role', 'p8_champion', 'p8_role', 'p9_champion', 'p9_role'])
    return df_matchs


## 4. Utilitaires d'Analyse et de Diagnostic

Les fonctions suivantes fournissent des analyses statistiques et diagnostiques des données :

In [13]:
def get_high_corr_pairs(df, param=0.99):
    """
    args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        param (float): seuil de corrélation au-dessus duquel une paire de features est considérée comme très corrélée (ex: 0.99 pour ne garder que les paires de features ayant une corrélation absolue supérieure à 0.99)
    return:
        DataFrame: matrice de corrélation de toutes les features numériques du dataframe df
        DataFrame: matrice de corrélation triangulaire supérieure (sans la diagonale) de toutes les features numériques du dataframe df
        list: liste des paires de features très corrélées, sous la forme de tuples (feature1, feature2) pour les paires ayant une corrélation absolue supérieure à param, avec feature1 et feature2 étant les noms des features correspondantes dans df
    goal:
        identifier les paires de features très corrélées dans le dataframe df en calculant la matrice de corrélation de toutes les features numériques du dataframe df, puis en extrayant la matrice de corrélation triangulaire supérieure (sans la diagonale) pour ne garder que les paires de features uniques, et enfin en renvoyant la liste des paires de features très corrélées, c'est à dire les paires de features ayant une corrélation absolue supérieure à param, pour pouvoir ensuite décider de supprimer une des features de chaque paire très corrélée
    """
    
    numeric_df = df.select_dtypes(include="number")
    if numeric_df.empty:
        print("Aucune colonne numérique trouvée pour la corrélation.")
        return pd.DataFrame(), pd.DataFrame(), []

    corr_matrix = numeric_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    high_corr = [
        (col, row)
        for col in upper.columns
        for row in upper.index
        if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > param
    ]

    print(f"Paires très corrélées : {high_corr}")
    print(f"Nombre de paires : {len(high_corr)}")
    return corr_matrix, upper, high_corr

def get_constant_and_quasi_constant_cols(df, quasi_param=0.99):
    """
    args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        quasi_param (float): seuil de fréquence au-dessus duquel une feature est considérée comme quasi-constante (ex: 0.99 pour considérer comme quasi-constante toute feature dont la modalité la plus fréquente représente plus de 99% des valeurs)
    return:
        list: liste des colonnes constantes, c'est à dire les colonnes du dataframe df qui ne contiennent qu'une seule valeur unique (en comptant les NaN comme une valeur possible)
        list: liste des colonnes quasi-constantes, c'est à dire les colonnes du dataframe df pour lesquelles la modalité la plus fréquente représente plus de quasi_param (ex: 99%) des valeurs, en comptant les NaN comme une modalité possible, et en excluant les colonnes contenant des listes ou des dictionnaires pour éviter les problèmes de calcul de fréquence liés à la nature non scalaire de ces types de données
    goal:
        identifier les colonnes constantes et quasi-constantes dans le dataframe df en vérifiant pour chaque colonne du dataframe df si elle ne contient qu'une seule valeur unique (en comptant les NaN comme une valeur possible) pour identifier les colonnes constantes, puis en calculant la fréquence de la modalité la plus fréquente de chaque colonne (en comptant les NaN comme une modalité possible) pour identifier les colonnes quasi-constantes, en excluant les colonnes contenant des listes ou des dictionnaires pour éviter les problèmes de calcul de fréquence liés à la nature non scalaire de ces types de données, et enfin en renvoyant la liste des colonnes constantes et quasi-constantes pour pouvoir ensuite décider de supprimer ces colonnes du dataframe df pour éviter les problèmes de surapprentissage (overfitting) liés à ces types de features peu ou pas du tout informatives
    """
    constant_cols = []
    quasi_constant_cols = []

    for col in df.columns:
        s = df[col]

        # On évite les colonnes contenant list/dict pour les stats de fréquence
        if s.apply(lambda x: isinstance(x, (list, dict))).any():
            continue

        if s.nunique(dropna=False) <= 1:
            constant_cols.append(col)
            continue

        vc = s.value_counts(normalize=True, dropna=False)
        if not vc.empty and quasi_param < vc.iloc[0] < 1:
            quasi_constant_cols.append(col)

    print(f"Constantes ({len(constant_cols)}) : {constant_cols}")
    print(f"Quasi-constantes ({len(quasi_constant_cols)}) : {quasi_constant_cols}")
    return constant_cols, quasi_constant_cols


def get_missing_cols(df, param=0):
    """
    args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        param (float): seuil de pourcentage de valeurs manquantes au-dessus duquel une colonne est considérée comme ayant beaucoup de valeurs manquantes
    return:
        list: liste des colonnes du dataframe df ayant plus de param*100% de valeurs manquantes (NaN), en comptant les NaN comme des valeurs manquantes, pour pouvoir ensuite décider de supprimer ces colonnes du dataframe df ou de les imputer pour éviter les problèmes liés aux valeurs manquantes dans ces colonnes, en fonction du pourcentage de valeurs manquantes et de l'importance potentielle de ces colonnes pour le modèle 
    goal:
        identifier les colonnes du dataframe df ayant beaucoup de valeurs manquantes en calculant le pourcentage de valeurs manquantes (NaN) de chaque colonne du dataframe df, en comptant les NaN comme des valeurs manquantes, et en renvoyant la liste des colonnes ayant plus de param*100% de valeurs manquantes pour pouvoir ensuite décider de supprimer ces colonnes du dataframe df ou de les imputer pour éviter les problèmes liés aux valeurs manquantes dans ces colonnes, en fonction du pourcentage de valeurs manquantes et de l'importance potentielle de ces colonnes pour le modèle
    """
    missing_cols = df.columns[df.isnull().mean() > param].tolist()
    print(f"Colonnes avec plus de {param*100}% de valeurs manquantes : {missing_cols}")
    return missing_cols



def categorize_features(df):
    """
    args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
    return:
        dict: dictionnaire catégorisant les features du dataframe df en fonction de leur type de données, avec comme clés les types de données ("bool", "list", "dict", "str", "int", "float", "other", "empty") et comme valeurs des listes de noms de colonnes du dataframe df correspondant à chaque type de données, en se basant à la fois sur les types de données déclarés dans le dataframe df et sur un échantillon de valeurs de chaque colonne pour détecter les types de données non scalaires (list, dict) ou les booléens masqués (ex: colonne de type object contenant uniquement les valeurs True/False), et en traitant les colonnes vides (colonnes dont toutes les valeurs sont NaN) comme une catégorie à part pour éviter les problèmes liés à l'inférence de type sur ces colonnes, pour pouvoir ensuite décider du type de traitement à appliquer à chaque feature en fonction de sa catégorie (ex: encodage pour les features booléennes ou catégorielles, imputation pour les features numériques avec des valeurs manquantes, etc.) et éviter les problèmes liés à un mauvais traitement des features en fonction de leur type de données
    goal:
        catégoriser les features du dataframe df en fonction de leur type de données en se basant à la fois sur les types de données déclarés dans le dataframe df et sur un échantillon de valeurs de chaque colonne pour détecter les types de données non scalaires (list, dict) ou les booléens masqués (ex: colonne de type object contenant uniquement les valeurs True/False), et en traitant les colonnes vides (colonnes dont toutes les valeurs sont NaN) comme une catégorie à part pour éviter les problèmes liés à l'inférence de type sur ces colonnes, et en renvoyant un dictionnaire catégorisant les features du dataframe df en fonction de leur type de données, avec comme clés les types de données ("bool", "list", "dict", "str", "int", "float", "other", "empty") et comme valeurs des listes de noms de colonnes du dataframe df correspondant à chaque type de données, pour pouvoir ensuite décider du type de traitement à appliquer à chaque feature en fonction de sa catégorie (ex: encodage pour les features booléennes ou catégorielles, imputation pour les features numériques avec des valeurs manquantes, etc.) et éviter les problèmes liés à un mauvais traitement des features en fonction de leur type de données
    """
    feature_categories = {
        "bool": [],
        "list": [],
        "dict": [],
        "str": [],
        "int": [],
        "float": [],
        "other": [],
        "empty": [],
    }

    for col in df.columns:
        sample = df[col].dropna()

        if sample.empty:
            feature_categories["empty"].append(col)
            continue

        first_val = sample.iloc[0]

        if isinstance(first_val, bool) or (
            df[col].dtype == object and sample.isin([True, False]).all()
        ):
            feature_categories["bool"].append(col)
        elif isinstance(first_val, list):
            feature_categories["list"].append(col)
        elif isinstance(first_val, dict):
            feature_categories["dict"].append(col)
        elif isinstance(first_val, str):
            feature_categories["str"].append(col)
        elif pd.api.types.is_bool_dtype(df[col]):
            feature_categories["bool"].append(col)
        elif pd.api.types.is_integer_dtype(df[col]):
            feature_categories["int"].append(col)
        elif pd.api.types.is_float_dtype(df[col]):
            feature_categories["float"].append(col)
        else:
            feature_categories["other"].append(col)

    for cat, cols in feature_categories.items():
        print(f"\n=== {cat.upper()} ({len(cols)}) ===")
        print(cols)

    non_numeric_cats = ["bool", "str", "list", "dict", "other"]
    cols_to_encode_or_check = [c for cat in non_numeric_cats for c in feature_categories[cat]]

    if cols_to_encode_or_check:
        print("Des features non numériques ont été détectées.")
        print("Encodage ou vérification a faire pour ces colonnes :")
        print(cols_to_encode_or_check)
    else:
        print("Toutes les features sont numériques (int/float).")

    return feature_categories

#fonction qui renvoie toutes les données des fonctions précédentes
def preprocess_start(df, corr_param=0.99, quasi_param=0.99, missing_param=0.9):
    """
    args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        corr_param (float): seuil de corrélation au-dessus duquel une paire de features est considérée comme très corrélée (ex: 0.99 pour ne garder que les paires de features ayant une corrélation absolue supérieure à 0.99)
        quasi_param (float): seuil de fréquence au-dessus duquel une feature est considérée comme quasi-constante (ex: 0.99 pour considérer comme quasi-constante toute feature dont la modalité la plus fréquente représente plus de 99% des valeurs)
        missing_param (float): seuil de pourcentage de valeurs manquantes au-dessus duquel une colonne est considérée comme ayant beaucoup de valeurs manquantes (ex: 0.9 pour considérer comme ayant beaucoup de valeurs manquantes toute colonne ayant plus de 90% de valeurs manquantes)
    return:
        dict: dictionnaire contenant les résultats de toutes les fonctions d'analyse du dataframe df, avec comme clés les noms des fonctions ("corr_matrix", "upper", "high_corr", "constant_cols", "quasi_constant_cols", "missing_cols", "feature_categories") et comme valeurs les résultats correspondants de chaque fonction, pour pouvoir avoir une vue d'ensemble de l'analyse du dataframe df et faciliter la prise de décision sur les traitements à appliquer à chaque feature en fonction de ces résultats (ex: suppression d'une des features de chaque paire très corrélée, suppression ou imputation des colonnes constantes ou quasi-constantes, suppression ou imputation des colonnes avec beaucoup de valeurs manquantes, traitement spécifique pour les features non numériques, etc.) et éviter les problèmes liés à un mauvais traitement des features en fonction de leur nature et de leur qualité
    goal:
        exécuter toutes les fonctions d'analyse du dataframe df (get_high_corr_pairs, get_constant_and_quasi_constant_cols, get_missing_cols, categorize_features) pour obtenir une vue d'ensemble de l'analyse du dataframe df, en utilisant les paramètres corr_param, quasi_param et missing_param pour personnaliser les seuils d'identification des paires de features très corrélées, des colonnes quasi-constantes et des colonnes avec beaucoup de valeurs manquantes, et en renvoyant un dictionnaire contenant les résultats de toutes ces fonctions pour faciliter la prise de décision sur les traitements à appliquer à chaque feature en fonction de ces résultats (ex: suppression d'une des features de chaque paire très corrélée, suppression ou imputation des colonnes constantes ou quasi-constantes, suppression ou imputation des colonnes avec beaucoup de valeurs manquantes, traitement spécifique pour les features non numériques, etc.) et éviter les problèmes liés à un mauvais traitement des features en fonction deleur nature et de leur qualité      
    """
    corr_matrix, upper, high_corr = get_high_corr_pairs(df, param=corr_param)
    constant_cols, quasi_constant_cols = get_constant_and_quasi_constant_cols(
        df, quasi_param=quasi_param
    )
    missing_cols = get_missing_cols(df, param=missing_param)
    feature_categories = categorize_features(df)

    return {
        "corr_matrix": corr_matrix,
        "upper": upper,
        "high_corr": high_corr,
        "constant_cols": constant_cols,
        "quasi_constant_cols": quasi_constant_cols,
        "missing_cols": missing_cols,
        "feature_categories": feature_categories,
    }




In [14]:
def drop_colonnes(df,*lists_colonnes):
    """args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        *lists_colonnes (list): une ou plusieurs listes de noms de colonnes à supprimer du dataframe df, ces listes peuvent être obtenues à partir des fonctions d'analyse du dataframe df (ex: liste des paires de features très corrélées, liste des colonnes constantes ou quasi-constantes, liste des colonnes avec beaucoup de valeurs manquantes, etc.) ou être définies manuellement en fonction de la connaissance du domaine ou des objectifs du projet
    return:
        DataFrame: dataframe df après suppression des colonnes dont les noms sont présents dans les listes de colonnes à supprimer, en vérifiant que ces colonnes existent bien dans le dataframe df avant de les supprimer pour éviter les erreurs liées à la tentative de suppression de colonnes inexistantes, et en renvoyant le dataframe df avec les colonnes supprimées pour pouvoir ensuite continuer le processus de préparation des données (ex: encodage des features non numériques, imputation des valeurs manquantes, etc.) sur ce dataframe nettoyé et éviter les problèmes liés à la présence de features peu ou pas du tout informatives ou redondantes dans le dataframe df
    goal:
        supprimer les colonnes du dataframe df dont les noms sont présents dans les listes de colonnes à supprimer, en vérifiant que ces colonnes existent bien dans le dataframe df avant de les supprimer pour éviter les erreurs liées à la tentative de suppression de colonnes inexistantes, et en renvoyant le dataframe df avec les colonnes supprimées pour pouvoir ensuite continuer le processus de préparation des données (ex: encodage des features non numériques, imputation des valeurs manquantes, etc.) sur ce dataframe nettoyé et éviter les problèmes liés à la présence de features peu ou pas du tout informatives ou redondantes dans le dataframe df
    """
    cols_to_drop = set()
    for lst in lists_colonnes:
        cols_to_drop.update(lst)

    cols_to_drop = [col for col in cols_to_drop if col in df.columns]

    return df.drop(columns = cols_to_drop)

In [15]:
def traiter_nan_lignes(df, seuil=10):
    """args:
        df (DataFrame): dataframe contenant les données de tous les matchs sélectionnés, avec une ligne par match et des colonnes pour les features de chaque participant à chaque minute sélectionnée
        seuil (int): nombre maximum de valeurs manquantes (NaN) autorisé dans une ligne du dataframe df, au-delà duquel la ligne est considérée comme ayant trop de valeurs manquantes et est supprimée du dataframe df pour éviter les problèmes liés à la présence de lignes avec beaucoup de valeurs manquantes qui peuvent créer un biais dans le modèle ou fausser les analyses statistiques, en fonction du nombre total de features du dataframe df et du pourcentage de valeurs manquantes que cela représente, et en renvoyant le dataframe df après suppression des lignes ayant plus de seuil valeurs manquantes pour pouvoir ensuite continuer le processus de préparation des données (ex: encodage des features non numériques, imputation des valeurs manquantes restantes, etc.) sur ce dataframe nettoyé et éviter les problèmes liés à la présence de lignes avec beaucoup de valeurs manquantes dans le dataframe df
    return:
        DataFrame: dataframe df après suppression des lignes ayant plus de seuil valeurs manquantes, en vérifiant que le seuil est cohérent avec le nombre total de features du dataframe df pour éviter de supprimer une grande partie des données, et en renvoyant le dataframe df avec les lignes supprimées pour pouvoir ensuite continuer le processus de préparation des données (ex: encodage des features non numériques, imputation des valeurs manquantes restantes, etc.) sur ce dataframe nettoyé et éviter les problèmes liés à la présence de lignes avec beaucoup de valeurs manquantes dans le dataframe df
    goal:
        traiter les lignes du dataframe df contenant des valeurs manquantes (NaN) en supprimant les lignes ayant plus de seuil valeurs manquantes pour éviter les problèmes liés à la présence de lignes avec beaucoup de valeurs manquantes qui peuvent créer un biais dans le modèle ou fausser les analyses statistiques, en fonction du nombre total de features du dataframe df et du pourcentage de valeurs manquantes que cela représente, et en renvoyant le dataframe df après suppression des lignes ayant plus de seuil valeurs manquantes pour pouvoir ensuite continuer le processus de préparation des données (ex: encodage des features non numériques, imputation des valeurs manquantes restantes, etc.) sur ce dataframe nettoyé et éviter les problèmes liés à la présence de lignes avec beaucoup de valeurs manquantes dans le dataframe df
    """
    df = df.copy()
    
    # garder seulement les lignes avec moins de seuil NaN
    mask = df.isna().sum(axis=1) < seuil
    df = df[mask]
    
    # remplacer les NaN restants par la médiane des colonnes
    df = df.fillna(df.median(numeric_only=True))
    
    return df

## Main
## 5. Pipeline Principal d'Exécution

Exécution du pipeline complet de preprocessing avec chargement des données, transformation et export :

In [16]:
events_to_keep = ['BUILDING_KILL','CHAMPION_KILL','ELITE_MONSTER_KILL','TURRET_PLATE_DESTROYED']
minutes_selected = [
    # Snapshots isolés — pour voir où la prédiction devient fiable
    5,#Très peu d'actions à 5mun donc features quasi-nulles donc pas forcément utile mais bon pour l'analyse
    10,
    15,
    20,
    25,
    # Combinaisons — pour voir la valeur d'ajouter du contexte temporel
    [10, 15],# début → mi
    [15, 20],# mi → mi-tard
    [10, 15, 20],# trajectoire early/mid (probablement le meilleur)
    [10, 20],# delta large
    [10, 15, 20, 25] # toute la partie
]


matchs_2_supp = matchs_interrompu(matchs_recap)
matchs_2_supp.update(match_less_than_time_needed(matchs_timeline,25)) # on prend le max de temps besoin pour chacun des matchs
matchs_recap,matchs_timeline = recup_matchs_without_pb(matchs_timeline,matchs_recap,matchs_2_supp)

for time in minutes_selected:
    print("########################################################################") 
    name = "_".join(map(str, time)) if isinstance(time, list) else str(time)
    print(f"Début du preprocessing du matchs_{name}")

    df_matchs = applatir_matchs(matchs_timeline,matchs_recap,puuids,time)

    df_matchs['game_rank'] = df_matchs.apply(encode_rank, axis=1)
    df_matchs = df_matchs.drop(columns=['rank', 'tier'])

    region_series = df_matchs["matchId"].apply(get_region)

    # One-Hot Encoding afin de ne pas rapprocher une region par rapport a une ature les 3 sont différentes et les playstyle différent égalements. 
    region_dummies = pd.get_dummies(region_series, prefix="region", dtype=int)
    #print(region_dummies.value_counts())
    # Ajout au dataframe
    df_matchs = pd.concat([df_matchs, region_dummies], axis=1)

    #Ajout win binaire
    df_matchs["win"] = (df_matchs["winningTeam"]==100).astype(int) #1 pour win de l'équipe blue, 0 pour win de l'équipe r
    df_matchs = df_matchs.drop(columns="winningTeam")

    df_matchs = encode_champ_role(df_matchs)

    preprocess_results = preprocess_start(df_matchs)
    df_matchs = drop_colonnes(df_matchs,preprocess_results["constant_cols"], preprocess_results["quasi_constant_cols"])

    df_matchs = traiter_nan_lignes(df_matchs)
    preprocess_results = preprocess_start(df_matchs)

    df_matchs.to_csv(f"matchs_preprocessed_{name}.csv")
    print(f"Enregistrements du matchs_{name} as matchs_preprocessed_{name}")
    print("########################################################################") 
print("FIN")

Ce match a eu une interruption inattendu : NA1_5496650813 
endOfGameResult = Abort_Unexpected
Ce match a eu une interruption inattendu : NA1_5509888764 
endOfGameResult = Abort_Unexpected
Ce match a eu une interruption inattendu : NA1_5496650813 
endOfGameResult = Abort_Unexpected
Ce match a eu une interruption inattendu : NA1_5509888764 
endOfGameResult = Abort_Unexpected
Nombre match interrompu : 4
type d'erreur rencontrée : {'Abort_Unexpected'}
Nombre match de moins de 25min : 898
matchs recap chargés : 5213
matchs timeline chargés : 5213
########################################################################
Début du preprocessing du matchs_5
wr_features ajoutées: 10
Features de Winrate ajoutées. Nouvelle shape : (5213, 186)
Paires très corrélées : []
Nombre de paires : 0
Constantes (20) : ['p0_buildings_killed_min5', 'p0_elite_monsters_killed_min5', 'p1_buildings_killed_min5', 'p1_elite_monsters_killed_min5', 'p2_buildings_killed_min5', 'p2_elite_monsters_killed_min5', 'p3_buildi